# WavLens: waveform + pyannote annotations + functions + playhead

Wheel = zoom, drag/scroll = pan, click = seek, space = play/pause.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import soundfile as sf
import torch
from pyannote.core import Annotation, Segment, SlidingWindow, SlidingWindowFeature, Timeline
from pyannote.database.util import load_rttm

import wavlens

path = "data/sample.wav"
x, sr = sf.read(path, dtype="float32")
reference = load_rttm("data/sample.rttm")["sample"]
x.shape, sr, reference.labels()

## Audio input forms

Path.

In [ ]:
wavlens.show(path, reference)

pyannote AudioFile dict with `"audio"`.

In [ ]:
wavlens.show({"audio": path}, reference)

pyannote AudioFile dict with `"waveform"` (C, T) tensor.

In [ ]:
wavlens.show({"waveform": torch.from_numpy(x)[None], "sample_rate": sr}, reference)

`(tensor, sr)`, i.e. what `torchaudio.load(path)` returns (CPU or GPU tensor).

In [ ]:
wavlens.show((torch.from_numpy(x)[None], sr), reference)

`sample_rate=` is an optional target rate: every audio is resampled to it.

In [ ]:
wavlens.show("data/sample.wav", (x[::2], 8000), sample_rate=16000)

Every array carries its rate as `(array, sr)` or `{"waveform": array, "sample_rate": sr}` (a bare array raises). Mixed rates without a target are resampled to the highest rate.

In [ ]:
wavlens.show((x, sr), (x[::2], 8000), reference)  # 8 kHz row resampled to 16 kHz

pyannote `Audio` loader output (not executed here: `pyannote.audio` 4.0.4's loader segfaults in this env).
```python
from pyannote.audio import Audio
wav, sr_a = Audio(sample_rate=16000, mono="downmix")(path)
wavlens.show({"waveform": wav, "sample_rate": sr_a}, reference)
```

Dict of named audios: one row each, played together (a list works too; unnamed rows become `Audio_0`, `Audio_1`).

In [ ]:
rng = np.random.default_rng(0)
wavlens.show(audio={"clean": (x, sr), "noisy": (x + 0.02 * rng.standard_normal(len(x)), sr), "quiet": (x * 0.3, sr)},
           annotation=reference)

## Annotation forms

`Timeline` (speech regions).

In [ ]:
wavlens.show(path, annotation=reference.get_timeline())

Single `Segment`.

In [ ]:
wavlens.show(path, annotation=Segment(5.0, 12.0))

File paths work for both audio and annotation: `.rttm` gives one group per uri, `.lab`/`.txt` are `start end label` lines, `.json` is the `DiarizeOutput.serialize()` shape. Files are read once per call, audio files in parallel.

In [ ]:
wavlens.show("data/sample.wav", "data/sample.rttm")

List: one lane group each.

In [ ]:
wavlens.show(path, annotation=[reference, Segment(1, 2)])

Dict of named lane groups.

In [ ]:
wavlens.show(path, annotation={"reference": reference,
                              "overlap": reference.get_overlap(),
                              "speaker90 only": reference.subset(["speaker90"])})

Fake hypothesis (reference shifted +0.3 s, relabeled) vs reference; shared colors per label.

In [ ]:
names = {lab: f"SPEAKER_{i:02d}" for i, lab in enumerate(reference.labels())}
hypothesis = Annotation(uri=reference.uri)
for seg, _, lab in reference.itertracks(yield_label=True):
    hypothesis[Segment(seg.start + 0.3, seg.end + 0.3)] = names[lab]

wavlens.show(path, annotation={"hypothesis": hypothesis, "reference": reference})

Named audio and annotation paths mixed with in-memory objects.

In [ ]:
wavlens.show(audio={"a": "data/sample.wav"}, annotation={"ref": "data/sample.rttm", "hyp": hypothesis})

Other diarization formats, as objects and as files: NeMo Sortformer lines, `(start, end, label)` tuples, records (`start`/`end` or SegLST `start_time`/`end_time`, `speaker`/`label`), pyannote `serialize()` dict, `DiarizeOutput`, RTTM text, and `.rttm` `.uem` `.lab` `.txt` `.json` `.seglst` `.jsonl` (NeMo manifest) paths.

In [ ]:
wavlens.show(path, ["0.00 4.98 speaker_0", "5.38 11.89 speaker_1"])  # NeMo Sortformer diarize() output

In [ ]:
wavlens.show(path, [(0.0, 4.98, "A"), (5.38, 11.89, "B"), (12.0, 15.0, "A")])  # (start, end, label) tuples

In [ ]:
wavlens.show(path, annotation={"records": [{"start": 0, "end": 1, "speaker": "S0"}, {"start": 1.5, "end": 4, "speaker": "S1"}],
                            "seglst": [{"start_time": 0.5, "end_time": 3.0, "speaker": "S0"}]})

In [ ]:
serialized = {"diarization": [{"start": s.start, "end": s.end, "speaker": lab}
                              for s, _, lab in reference.itertracks(yield_label=True)]}  # pyannote serialize() shape
wavlens.show(path, serialized)

In [ ]:
wavlens.show(path, open("data/sample.rttm").read())  # RTTM text

In [ ]:
open("data/sample.uem", "w").write("sample 1 0.0 12.5\nsample 1 20.0 30.0\n")
wavlens.show(path, "data/sample.uem")

In [ ]:
open("data/sample.lab", "w").write("0.00 4.98 speaker_0\n5.38 11.89 speaker_1\n")
wavlens.show(path, "data/sample.lab")

In [ ]:
import json
json.dump([{"start_time": 0.0, "end_time": 4.98, "speaker": "spk0"}, {"start_time": 5.38, "end_time": 11.89, "speaker": "spk1"}],
          open("data/sample.seglst", "w"))
wavlens.show(path, "data/sample.seglst")

In [ ]:
open("data/sample.jsonl", "w").write(json.dumps({"audio_filepath": "sample.wav", "rttm_filepath": "sample.rttm"}) + "\n")  # NeMo manifest; paths relative to the manifest
wavlens.show(path, "data/sample.jsonl")

Error lane from `pyannote.metrics` identification error analysis; `colors=` keyed by annotation name, then by speaker.

In [ ]:
from pyannote.metrics.errors.identification import IdentificationErrorAnalysis

diff = IdentificationErrorAnalysis().difference(reference, hypothesis)
err = Annotation(uri=reference.uri)
for seg, _, (status, r, h) in diff.itertracks(yield_label=True):
    err[seg] = status

wavlens.show(path, annotation={"reference": reference, "hypothesis": hypothesis, "errors": err},
           colors={"errors": {"correct": "#4daf4a", "confusion": "#984ea3",
                              "missed detection": "#e41a1c", "false alarm": "#ff7f00"}})

`colors=` mirrors the `annotation` form (dict → dict keyed by annotation name, subset ok; list → list per annotation; one → one). Each entry is `{speaker: css}` by name, or a list of css colors in that annotation's `labels()` order.

In [ ]:
wavlens.show(path, annotation=[reference, hypothesis], colors=[["#1b9e77", "#d95f02"], {"SPEAKER_00": "#7570b3"}])

## Functions

Single 1-D signal: RMS envelope (min-max scaled, drawn over every row).

In [ ]:
win = 400
rms = np.sqrt(np.convolve(x ** 2, np.ones(win) / win, mode="same"))
wavlens.show(path, reference, function=rms)

Dict of named signals.

In [ ]:
frame = 160
n = len(x) // frame
zcr = (np.diff(np.sign(x[: n * frame]).reshape(n, frame), axis=1) != 0).mean(1)
wavlens.show(path, reference, function={"rms": rms, "zcr": zcr})

pyannote `SlidingWindowFeature` (25 ms frames, 10 ms step): real time axis.

In [ ]:
sw = SlidingWindow(start=0.0, duration=0.025, step=0.010)
starts = np.arange(0, len(x) / sr - sw.duration, sw.step)
idx = (starts[:, None] * sr).astype(int) + np.arange(int(sw.duration * sr))
energy = (x[idx] ** 2).mean(1)[:, None]
wavlens.show(path, reference, function={"energy": SlidingWindowFeature(energy, sw)})

`SlidingWindowFeature` with 2 columns: one line per column.

In [ ]:
feat2 = SlidingWindowFeature(np.stack([energy[:, 0], np.sqrt(energy[:, 0])], 1), sw)
wavlens.show(path, reference, function={"energy": feat2})

## Functions per audio

Dict keys equal to an audio name draw only on that row (one function, a list, or a dict); other keys draw on every row.

In [ ]:
noise = 0.02 * rng.standard_normal(len(x))
wavlens.show(audio={"clean": (x, sr), "noisy": (x + noise, sr)}, annotation=reference,
           function={"clean": rms, "noisy": [rms, zcr], "global": rms[::100]})

A 40-point function stretched over the whole duration with `interpolation="previous"` (stepwise; also `"nearest"`, `"linear"`, `"cubic"`).

In [ ]:
coarse = rms[:: len(rms) // 40][:40]
wavlens.show(path, reference, function={"coarse": coarse}, interpolation="previous")

## Layout

Many annotations and speakers: the lane block grows with the rows (`row_height` px each). A click on a name in the left column cycles neutral → keep its segments → keep the complement → neutral (audio outside the kept part is silenced, whited out and skipped); several active names intersect.

In [ ]:
rng = np.random.default_rng(0)
many = {}
for a in range(5):
    ann = Annotation(); t = 0.0
    while t < 29:
        d = rng.uniform(0.4, 2.0); ann[Segment(t, t + d)] = f"spk{rng.integers(8)}"; t += d + rng.uniform(0, 0.5)
    many[f"ann_{a}"] = ann
wavlens.show("data/sample.wav", annotation=many, legend="both")

`height=160`.

In [ ]:
wavlens.show(path, reference, height=160)

`row_height=28` with 3 annotations.

In [ ]:
wavlens.show(path, annotation={"reference": reference, "hypothesis": hypothesis, "errors": err}, row_height=28)

`legend="both"` (or `"side"`), with a `colors=` override for one speaker.

In [ ]:
wavlens.show(path, annotation={"reference": reference, "hypothesis": hypothesis}, legend="both",
           colors={"reference": {"speaker90": "#222"}})

`step=0.1`: seconds per ◀ / ▶ press.

In [ ]:
wavlens.show(path, reference, step=0.1)

## Spectrogram

`view="fft"` opens on the spectrogram; `colormap=` is any matplotlib name. Pick one you like.

`magma`

In [ ]:
wavlens.show(path, reference, view="fft", colormap="magma", height=80)

`inferno`

In [ ]:
wavlens.show(path, reference, view="fft", colormap="inferno", height=80)

`viridis`

In [ ]:
wavlens.show(path, reference, view="fft", colormap="viridis", height=80)

`turbo`

In [ ]:
wavlens.show(path, reference, view="fft", colormap="turbo", height=80)

`hot`

In [ ]:
wavlens.show(path, reference, view="fft", colormap="hot", height=80)

`gray_r`

In [ ]:
wavlens.show(path, reference, view="fft", colormap="gray_r", height=80)

Frequency axis: `fscale="mel"` is the default (the low band gets most rows); `fscale="linear"` is available.

In [ ]:
wavlens.show("data/sample.wav", reference, view="fft", fscale="linear", height=120)

`fmax=` caps the frequency axis (default Nyquist); 5000 Hz is a good speech setting.

In [ ]:
wavlens.show("data/sample.wav", reference, view="fft", fmax=5000, height=120)

## Weights and interpolation

Without weights the mix is 1/n over the enabled audios and the ☑ toggles re-normalize live. With weights you see and hear w·audio, and toggles only mute.

Dict audio + dict of scalar weights.

In [ ]:
wavlens.show(audio={"speech": (x, sr), "noise": (noise, sr)}, annotation=reference,
           weights={"speech": 0.9, "noise": 0.1})

Elementwise weight of the audio's length: a fade-in on one audio, scalar 1 on the other.

In [ ]:
wavlens.show(audio={"speech": (x, sr), "noise": (noise, sr)}, annotation=reference,
           weights={"speech": np.linspace(0, 1, len(x)), "noise": 1})

Short envelope interpolated to the audio: `interpolation="linear"`.

In [ ]:
env = np.array([0, 1, 1, 0, 1])
wavlens.show(audio={"speech": (x, sr), "noise": (noise, sr)}, annotation=reference,
           weights={"speech": env, "noise": 0.2}, interpolation="linear")

`interpolation="previous"` (stepwise).

In [ ]:
wavlens.show(audio={"speech": (x, sr), "noise": (noise, sr)}, annotation=reference,
           weights={"speech": env, "noise": 0.2}, interpolation="previous")

List audio + list weights.

In [ ]:
wavlens.show(audio=[(x, sr), (x * 0.3, sr)], annotation=reference, weights=[1, 2])

Single positional audio + scalar weight.

In [ ]:
wavlens.show((x, sr), reference, weights=0.5)

## Control from Python

In [ ]:
w = wavlens.show(audio={"a": (x, sr), "b": (x * 0.5, sr)}, annotation=reference)
w

`w.current_time` and `w.playing` update while playing; `w.enabled` mutes rows.

In [ ]:
w.seek(10)
w.enabled = [True, False]
w.current_time, w.playing, w.enabled

Keys on the focused widget: space = play/pause, ← / → = step. Buttons: ◀ ▶, all/none, fft, legend, ☑ per audio.

Every UI state is an argument: `enabled`, `solo`, `dim`, `zoom`, `time`, `autoplay`. `solo` takes `'ann'`, `'ann/speaker'`, `'!ann/speaker'` for the complement, or a list that intersects; `dim` has no click gesture, only the argument and `w.dim(...)`.

In [ ]:
wavlens.show("data/sample.wav", annotation={"ref": reference}, solo="ref", zoom=200, time=5.0, legend="both")  # keep the annotation

In [ ]:
wavlens.show("data/sample.wav", annotation={"ref": reference}, solo="!ref/speaker91", legend="both")  # keep the complement

In [ ]:
wavlens.show("data/sample.wav", annotation={"ref": reference}, solo=["ref", "!ref/speaker91"], legend="both")  # intersection

Same from Python after the fact: `w.solo_(...)` (underscore: `solo` is the trait; no arguments clears) and `w.dim(...)`.

In [ ]:
w = wavlens.show("data/sample.wav", annotation={"ref": reference}, legend="both")
w.solo_("ref", "!ref/speaker91"); w.dim("ref/speaker91")
w.solo, w.dimmed

## Real pipeline

```python
from pyannote.audio import Pipeline
pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-community-1")  # local copy under ~/nemo/nemo-models/hf/
out = pipeline(path)
wavlens.show(path, annotation={"hyp": out.speaker_diarization, "exclusive": out.exclusive_speaker_diarization, "ref": reference})
```
Idea: pass `hook=` to the pipeline to capture `speaker_counting` and show it as a `function=`.